# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashidsami10000-afk/rashid-flyrank-internship-ml-owncopy/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane 4 (CTR / Engagement Opportunity Scoring): does a learned model beat the Week-4 hand rule
(`ctr_below_tier_benchmark`, P@50 = 0.920) at ranking pages that under-capture their position
tier next month?

Same data as ML-04/07 (one row = one client × page over March 2026; five contract features;
April used only for labels), same metric family (Precision@K with base rate printed), same
client-grouped validation philosophy — everything compared on identical splits.

## 0. Setup — rebuild the exact ML-07 frame

In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [ ]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

import pathlib

def find_repo_root():
    p = pathlib.Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / 'work' / 'notebooks').exists():
            return cand
    return p

ROOT = find_repo_root()
OUT_DIR = ROOT / 'work' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'outputs -> {OUT_DIR}')

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

q_mar = f"""
    WITH pagemo AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)  AS imp_mar,
               SUM(gsc_clicks)       AS clk_mar,
               AVG(gsc_avg_position) AS pos_mar
        FROM {MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND AVG(gsc_avg_position) > 0
    ),
    climpo AS (
        SELECT client_hash_id, SUM(gsc_impressions) AS cli_imp
        FROM {MAR}
        GROUP BY 1
    )
    SELECT p.*, c.cli_imp
    FROM pagemo p JOIN climpo c USING (client_hash_id)
"""
mar = con.sql(q_mar).df()

q_apr = f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_apr,
           SUM(gsc_clicks)      AS clk_apr
    FROM {APR}
    GROUP BY 1, 2
"""
apr = con.sql(q_apr).df()

def tier_of(pos):
    if pos <= 3:
        return 'p1_top'
    if pos <= 10:
        return 'p1'
    if pos <= 20:
        return 'p2'
    return 'deep'

frame = mar.merge(apr, on=['client_hash_id', 'content_hash_id'], how='inner').copy()
frame['tier'] = frame['pos_mar'].apply(tier_of)
frame['ctr_mar'] = frame['clk_mar'] / frame['imp_mar']
frame['apr_ctr'] = np.where(frame['imp_apr'] > 0, frame['clk_apr'] / frame['imp_apr'], np.nan)

bench = frame[frame['imp_mar'] >= 1000].groupby('tier')['ctr_mar'].median().rename('tier_expected_ctr')
frame['tier_expected_ctr'] = frame['tier'].map(bench)
bench_apr = frame[frame['imp_apr'] >= 1000].groupby('tier')['apr_ctr'].median().rename('tier_expected_apr')
frame['tier_expected_apr'] = frame['tier'].map(bench_apr)

labeled = frame[(frame['imp_apr'] >= 100) & (frame['tier_expected_ctr'] > 0) &
                (frame['tier_expected_apr'] > 0)].copy()
labeled['under_captured_apr'] = (
    (labeled['apr_ctr'] / labeled['tier_expected_apr']) < 0.5).astype(int)

# five contract features (ML-04) + the Week-4 rule score
labeled['log10_imp_mar'] = np.log10(labeled['imp_mar'])
labeled['imp_share_client'] = labeled['imp_mar'] / labeled['cli_imp']
labeled['capture_ratio'] = labeled['ctr_mar'] / labeled['tier_expected_ctr']
labeled['rule_score'] = (100 * (1 - labeled['capture_ratio']).clip(lower=0)
                         * np.log10(labeled['imp_mar']))

FEATURES = ['ctr_mar', 'pos_mar', 'log10_imp_mar', 'tier_expected_ctr', 'imp_share_client']
print(f'labeled frame: {len(labeled):,} rows, {labeled["client_hash_id"].nunique():,} clients')
print(f'label base rate: {labeled["under_captured_apr"].mean():.3f}')
print(f'features: {FEATURES} + rule_score (baseline, not a model feature)')

## 1. Method choice and why

The lane's question shape is *"which pages first?"* — a ranking problem. Per this week's
toolkit table: any classifier's **probability**, evaluated at **precision@K**, against the
hand-rule baseline. Three methods, ordered by complexity:

| Method | Why it is here |
|---|---|
| **Logistic Regression** (primary) | Readable coefficients, monotone ranking scores, well-calibrated probabilities; the natural "learned version" of the hand rule on the same five features. |
| **Decision Tree** (depth 3) | Printable rules — shows whether an *interpretable* model already captures the pattern, and gives the error section concrete thresholds to argue with. |
| **Gradient Boosting** | The complexity test: only earns its place if the comparison table shows a real gain over LR. |

Not chosen: **clustering** (the lane outputs a ranked queue, not groups), **correlation-only
analysis** (the signal audit in weeks 3–4 already established the signals), heavier ensembles
beyond one GB (nothing here suggests the extra machinery pays). Seeds fixed at 42 everywhere.

In [ ]:
import sklearn
print('reproducibility receipt:')
print(f'  scikit-learn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__}')
print(f'  SEED={SEED} fixed for splits, models, permutation importance, random floor')

## 2. Split design

**Client-grouped holdout (25%), seed 42** — the same protocol ML-03/04/07 used. Pages from one
client share hidden character (template, authority, analytics setup); a random split lets a
model memorize the client and fake skill. The honest question is *"does it rank pages of a
client it never saw?"*

**Time is handled by construction, not by splitting inside the notebook:** every feature comes
from March 2026, every label from April 2026 — one strict step ahead. There is no future
information to leak through the split because no feature window overlaps the label window.

On top of the single holdout, a **5-fold client-grouped cross-check** reports how much the
headline numbers wander across different client partitions — one split can flatter anyone.

Known carry-over limitation (disclosed in ML-04 §4): the labeled frame keeps only pages that
still earn ≥100 April impressions — survivorship in the population, disclosed, not hidden.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
tr_idx, te_idx = next(gss.split(labeled, groups=labeled['client_hash_id']))
train_df = labeled.iloc[tr_idx]
test_df = labeled.iloc[te_idx]

print(f'train: {len(train_df):,} rows / {train_df["client_hash_id"].nunique():,} clients')
print(f'test : {len(test_df):,} rows / {test_df["client_hash_id"].nunique():,} clients')
overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print(f'clients in BOTH sides: {len(overlap)}  <- must be 0')

gkf = GroupKFold(n_splits=5)
fold_sizes = [(ftr.shape[0], fte.shape[0]) for ftr, fte in gkf.split(labeled, groups=labeled['client_hash_id'])]
print(f'5-fold grouped sizes (train, test): {fold_sizes}')

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:min(k, len(labels))]].mean())

y_te = test_df['under_captured_apr'].to_numpy()
print()
print(f'test base rate: {y_te.mean():.3f}')
print(f'random floor on this test set: '
      f'P@20 {p_at_k(rng.random(len(y_te)), y_te, 20):.3f} | '
      f'P@50 {p_at_k(rng.random(len(y_te)), y_te, 50):.3f}')

## 3. Train + compare vs my baseline

Same data, same metric, same split. The Week-4 rule needs no training — its score is computed
on every row including the test set. Models see **train rows only**; predictions are made on
the untouched test set.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

models = {
    'Logistic Regression': make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    'Decision Tree (depth 3)': DecisionTreeClassifier(max_depth=3, random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(random_state=SEED),
}

X_tr, y_tr = train_df[FEATURES], train_df['under_captured_apr']
for name, mdl in models.items():
    mdl.fit(X_tr, y_tr)
    print(f'{name}: fitted on {len(X_tr):,} train rows')

In [ ]:
rows = []
labels_col = 'under_captured_apr'

def add_row(name, scores, df):
    y = df[labels_col].to_numpy()
    rows.append({'method': name,
                 'P@20': round(p_at_k(scores, y, 20), 3),
                 'P@50': round(p_at_k(scores, y, 50), 3)})

add_row('base rate (always 1)', np.ones(len(test_df)), test_df)
add_row('random queue (seeded)', rng.random(len(test_df)), test_df)
add_row('Week-4 rule (baseline)', test_df['rule_score'], test_df)
for name, mdl in models.items():
    add_row(name, mdl.predict_proba(test_df[FEATURES])[:, 1], test_df)

table = pd.DataFrame(rows)
display(table)

best = table[table['method'].str.contains('Regression|Boosting|Tree')]['P@50'].max()
print(f'best model P@50 on holdout : {best:.3f}')
print(f'baseline rule P@50         : {table.loc[table.method == "Week-4 rule (baseline)", "P@50"].iloc[0]:.3f}')

In [ ]:
# stability: 5-fold client-grouped CV for rule vs the two serious contenders
cv_rows = []
for fold, (ftr, fte) in enumerate(gkf.split(labeled, groups=labeled['client_hash_id']), 1):
    tr_d, te_d = labeled.iloc[ftr], labeled.iloc[fte]
    lr = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=2000, random_state=SEED)).fit(
                       tr_d[FEATURES], tr_d[labels_col])
    gb = GradientBoostingClassifier(random_state=SEED).fit(tr_d[FEATURES], tr_d[labels_col])
    cv_rows.append({
        'fold': fold,
        'rule_P50': round(p_at_k(te_d['rule_score'], te_d[labels_col], 50), 3),
        'lr_P50': round(p_at_k(lr.predict_proba(te_d[FEATURES])[:, 1], te_d[labels_col], 50), 3),
        'gb_P50': round(p_at_k(gb.predict_proba(te_d[FEATURES])[:, 1], te_d[labels_col], 50), 3),
        'base_rate': round(te_d[labels_col].mean(), 3),
    })
cv_table = pd.DataFrame(cv_rows)
display(cv_table)
print(cv_table[['rule_P50', 'lr_P50', 'gb_P50']].agg(['mean', 'std']).round(3))

In [ ]:
# receipts
import json, datetime
metrics = {
    'notebook': 'w05_model.ipynb',
    'generated_at_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'),
    'sklearn_version': sklearn.__version__,
    'seed': SEED,
    'split': 'GroupShuffleSplit(test_size=0.25, seed=42) by client_hash_id',
    'n_train': int(len(train_df)), 'n_test': int(len(test_df)),
    'holdout': table.to_dict(orient='records'),
    'cv5': cv_table.to_dict(orient='records'),
}
with open(OUT_DIR / 'w05_model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'wrote {OUT_DIR / "w05_model_metrics.json"}')

## 4. Errors and interpretation

What the winner leans on (permutation importance — checked by shuffling, not fitted weights),
then where it is most wrong at the top of the queue, then three concrete hard cases.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, make_scorer

final_name = max(models, key=lambda n: p_at_k(
    models[n].predict_proba(test_df[FEATURES])[:, 1], y_te, 50))
final_model = models[final_name]
print(f'method carried into error analysis: {final_name}')

ap_scorer = make_scorer(average_precision_score, needs_proba=True, greater_is_better=True)
pi = permutation_importance(final_model, test_df[FEATURES], test_df[labels_col],
                            scoring=ap_scorer, n_repeats=10, random_state=SEED)
imp_table = pd.DataFrame({'feature': FEATURES,
                          'importance_mean': pi.importances_mean.round(4),
                          'importance_std': pi.importances_std.round(4)}
                         ).sort_values('importance_mean', ascending=False)
display(imp_table.reset_index(drop=True))

if 'Logistic Regression' == final_name:
    lr_clf = final_model.named_steps['logisticregression']
    coefs = pd.DataFrame({'feature': FEATURES,
                          'coef': lr_clf.coef_[0].round(3)},
                         ).sort_values('coef', ascending=False)
    display(coefs.reset_index(drop=True))

In [ ]:
test_sorted = test_df.copy()
test_sorted['model_proba'] = final_model.predict_proba(test_df[FEATURES])[:, 1]
top50 = test_sorted.sort_values('model_proba', ascending=False).head(50)

fp = top50[top50[labels_col] == 0]
tp = top50[top50[labels_col] == 1]
print(f'top-50 by model: {len(tp)} true under-performers, {len(fp)} false alarms '
      f'(P@50 = {len(tp)/50:.2f})')
print()
print('false-alarm profile vs the rest of the test set:')
profile = pd.DataFrame({
    'false alarms (top-50, label=0)': fp[['capture_ratio', 'pos_mar', 'imp_mar']].median(),
    'all other test rows': test_sorted.drop(top50.index)[['capture_ratio', 'pos_mar', 'imp_mar']].median(),
})
display(profile)

boundary = (np.abs(fp['pos_mar'] - 10) < 1.0) | (np.abs(fp['pos_mar'] - 20) < 1.0)
print(f'false alarms sitting within 1 position of a tier boundary: {int(boundary.sum())} of {len(fp)}')

In [ ]:
hard = fp.head(3).copy()
hard['content_short'] = hard['content_hash_id'].str[:12]
hard['client_short'] = hard['client_hash_id'].str[:10]

def why_hard(r):
    bits = []
    if r['capture_ratio'] > 0.7:
        bits.append(f"already captures {r['capture_ratio']:.0%} of its benchmark - the gap is small")
    if abs(r['pos_mar'] - 10) < 1 or abs(r['pos_mar'] - 20) < 1:
        bits.append(f"position {r['pos_mar']:.1f} sits near a tier boundary, benchmark unstable")
    if r['imp_mar'] < 250:
        bits.append(f"only {int(r['imp_mar']):,} impressions - April CTR is noisy coin-flip territory")
    if not bits:
        bits.append('March gap looked real but April CTR recovered on its own (mean reversion)')
    return '; '.join(bits)

hard['why_hard'] = hard.apply(why_hard, axis=1)
with pd.option_context('display.max_colwidth', 80, 'display.width', 220):
    display(hard[['content_short', 'client_short', 'tier', 'pos_mar', 'imp_mar',
                  'capture_ratio', 'model_proba', labels_col, 'why_hard']])

overlap50 = len(set(top50.index) &
                set(test_sorted.sort_values('rule_score', ascending=False).head(50).index))
print(f'overlap between model top-50 and rule top-50: {overlap50} of 50')

### What the errors say

*(filled after execution — whether the model actually beat the rule or merely matched it,
which features carry the ranking and whether each passes the straight-face test, what kind of
page the queue gets wrong, and whether Gradient Boosting earned its complexity.)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.